# LIT Training — Google Colab

**Before running:** Go to `Runtime > Change runtime type` and set Hardware accelerator to **T4 GPU**.

**One-time setup:** Upload these files from your local machine to `My Drive/LIT/` in Google Drive:
```
My Drive/LIT/
  data/
    processed/
      hmm.pt
      vocab.json
      morfessor.model
    splits/
      train.json
      val.json
      test.json
      hmm_train.txt
```
Checkpoints will be saved back to `My Drive/LIT/checkpoints/` automatically.

In [ ]:
# Step 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2 — Clone repo and install dependencies
!git clone https://github.com/hynoph/ling-informed-transformers.git
%cd ling-informed-transformers/lit/LIT
!pip install -q torch morfessor pyyaml requests lxml tqdm numpy

In [ ]:
# Step 3 — Copy data files from Drive into the repo
import shutil, os

DRIVE_LIT = '/content/drive/MyDrive/LIT'

pairs = [
    (f'{DRIVE_LIT}/data/processed/hmm.pt',          'data/processed/hmm.pt'),
    (f'{DRIVE_LIT}/data/processed/vocab.json',       'data/processed/vocab.json'),
    (f'{DRIVE_LIT}/data/processed/morfessor.model',  'data/processed/morfessor.model'),
    (f'{DRIVE_LIT}/data/splits/train.json',          'data/splits/train.json'),
    (f'{DRIVE_LIT}/data/splits/val.json',            'data/splits/val.json'),
    (f'{DRIVE_LIT}/data/splits/test.json',           'data/splits/test.json'),
    (f'{DRIVE_LIT}/data/splits/hmm_train.txt',       'data/splits/hmm_train.txt'),
]

for src, dst in pairs:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(src, dst)
    print(f'Copied {os.path.basename(src)}')

print('All data files ready.')

In [ ]:
# Step 4 — Confirm GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# Step 5 — Train (runs all 30 epochs, ~1-3 min each on T4)
# Per-epoch checkpoints are saved to logs/lit_run/epoch_NNN.pt
# If this cell disconnects, re-run Step 3 then use --resume below instead.
!python run.py --config configs/data_config.yaml --skip_hmm

In [ ]:
# Step 5b — Resume from best checkpoint (use this if the session disconnected)
# First re-run Steps 1-3, then run this cell instead of Step 5.
!python run.py --config configs/data_config.yaml --skip_hmm --resume

In [ ]:
# Step 6 — Save checkpoints back to Drive
import shutil, os, glob

CKPT_DRIVE = '/content/drive/MyDrive/LIT/checkpoints'
os.makedirs(CKPT_DRIVE, exist_ok=True)

for pt in glob.glob('logs/lit_run/*.pt'):
    shutil.copy(pt, CKPT_DRIVE)
    print(f'Saved {os.path.basename(pt)} -> Drive')

shutil.copy('logs/lit_run/lit_run.log', CKPT_DRIVE)
print('Saved lit_run.log -> Drive')